# Homework 2 — Vector Search

LLM Zoomcamp 2026. Q1–Q6 were executed by Gerhard on 17 October 2026 (Europe/Zurich); saved code, outputs and execution counts are preserved. Publication review adds explanatory markdown without rerunning the exercises.

Dataset: 72 lesson pages at course commit `8c1834d114754cc0e0d65544b8589ef7d94b81cf`, preserving the `/lessons/` filter and original filenames.
Model: `Xenova/all-MiniLM-L6-v2` through the official CPU ONNX helper; 384-dimensional normalized vectors.

[Homework requirements](https://github.com/DataTalksClub/llm-zoomcamp/blob/1c42efbeb8141c183e7ae51698ad912c70c06a06/cohorts/2026/homework/02-vector-search/homework.md) · [Module 2 lessons](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/02-vector-search)

Course submission has not been performed or verified. See the final answer summary below.


## Environment check

The following learner-run cell records the project interpreter, working directory and dependency versions. Select the **LLMZ Homework 2** kernel to reproduce the notebook.


In [1]:
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Working directory:", Path.cwd())

for package in ("onnxruntime", "tokenizers", "numpy", "tqdm", "minsearch", "gitsource", "huggingface-hub", "ipykernel"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: 3.12.3
Interpreter: /workspaces/llmz-02-vector-search/.venv/bin/python
Working directory: /workspaces/llmz-02-vector-search
onnxruntime: 1.31.0
tokenizers: 0.23.3
numpy: 2.5.4
tqdm: 4.70.1
minsearch: 0.2.0
gitsource: 0.0.5
huggingface-hub: 2.2.0
ipykernel: 7.4.0


## Q1 — Embedding a query

Embed the exact question below. The vector has 384 coordinates; the first coordinate is `-0.020582046681707114`, so the closest option is **-0.02**. The saved ONNX virtual-device discovery warning did not prevent successful encoding.


In [2]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print("Vector shape:", v.shape)
print("First coordinate:", v[0])

2026-10-17 08:43:01.204503782 [W:onnxruntime:Default, device_discovery.cc:95 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Vector shape: (384,)
First coordinate: -0.020582046681707114


## Load the pinned lesson pages

Load the homework's 72 markdown pages from the full pinned commit. Preserve original filenames and the `/lessons/` filter. The target for Q2 is `02-vector-search/lessons/07-sqlitesearch-vector.md`.


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d114754cc0e0d65544b8589ef7d94b81cf",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

print("Lesson pages:", len(documents))
print("Q2 matches:", [
    doc["filename"] for doc in documents
    if doc["filename"] == target_filename
])

Lesson pages: 72
Q2 matches: ['02-vector-search/lessons/07-sqlitesearch-vector.md']


In [4]:
page = next(
    doc for doc in documents
    if doc["filename"] == target_filename
)

print("Filename:", page["filename"])
print("Characters:", len(page["content"]))
print("\nPreview:\n", page["content"][:500])

Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Characters: 7219

Preview:
 # Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over ev


## Q2 — Cosine similarity

The two vectors have unit length, so their dot product equals cosine similarity. The whole-page result is `0.3610702427672444`; closest option: **0.37**. The original `@` cell is preserved; the following learner-run `.dot()` cell uses the lesson's notation and confirms the same result.


In [5]:
import numpy as np

page_vector = embedder.encode(page["content"])

print("Page vector shape:", page_vector.shape)
print("Query vector length:", np.linalg.norm(v))
print("Page vector length:", np.linalg.norm(page_vector))
print("Cosine similarity:", v @ page_vector)

Page vector shape: (384,)
Query vector length: 0.9999999999999999
Page vector length: 1.0
Cosine similarity: 0.3610702427672444


In [6]:
similarity = v.dot(page_vector)

print("Cosine similarity using .dot():", similarity)

Cosine similarity using .dot(): 0.3610702427672444


## Q3 — Chunking and search by hand

Use the homework's `size=2000`, `step=1000` character windows. The 72 pages produce 295 chunks. Following the dataset-embedding lesson, encode batches of 50 and build `X`, one row per chunk and 384 columns. The homework's ONNX helper uses `encode_batch()` for chunk `content`.

`X.dot(v)` scores all chunks against the original Q1 vector, which remains in `v` even when later cells reuse the name `query`. Best chunk: index **94**, similarity **0.6489018052329629**, from **02-vector-search/lessons/07-sqlitesearch-vector.md**.


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print("Chunks:", len(chunks))
print("First chunk filename:", chunks[0]["filename"])

Chunks: 295
First chunk filename: 01-agentic-rag/lessons/01-intro.md


In [8]:
from tqdm.auto import tqdm
import numpy as np

texts = []

for doc in chunks:
    texts.append(doc["content"])

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embedder.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print("Embedding matrix shape:", X.shape)

  0%|          | 0/6 [00:00<?, ?it/s]

Embedding matrix shape: (295, 384)


In [9]:
scores = X.dot(v)

idx = np.argmax(scores)

print("Best chunk index:", idx)
print("Cosine similarity:", scores[idx])
print("Filename:", chunks[idx]["filename"])

Best chunk index: 94
Cosine similarity: 0.6489018052329629
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4 — Vector search with minsearch

Following the minsearch lesson, `fit(X, chunks)` pairs each embedding row with its chunk payload. No keyword filtering is required. The first result for the homework's evaluation query is **04-evaluation/lessons/05-search-metrics.md**.


In [10]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=[])
vindex.fit(X, chunks)

print("Indexed chunks:", len(chunks))

Indexed chunks: 295


In [11]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

results = vindex.search(query_vector, num_results=5)

print("First result filename:", results[0]["filename"])

First result filename: 04-evaluation/lessons/05-search-metrics.md


## Q5 — Text search versus vector search

Create a lexical index over the same chunk `content`, then search both indexes with the exact PostgreSQL query and five results each. Vector search receives an embedding; text search receives the query string. Repeated filenames represent different chunks.

**02-vector-search/lessons/08-pgvector.md** appears among the vector results and is absent from the text results.


In [12]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=[]
)

index.fit(chunks)

print("Text-indexed chunks:", len(chunks))

Text-indexed chunks: 295


In [13]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embedder.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
text_results = index.search(query, num_results=5)

print("Vector search:")
for doc in vector_results:
    print(doc["filename"])

print("\nText search:")
for doc in text_results:
    print(doc["filename"])

Vector search:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

Text search:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


## Q6 — Hybrid search with Reciprocal Rank Fusion

Use the homework's supplied RRF function unchanged apart from the confirmation print. It adds `1 / (60 + rank)` for each list containing a chunk, with zero-based ranks. `(filename, start)` identifies the chunk. Both candidate lists contain their top five chunks.

For the exact tool-access query, vector search starts with `01-agentic-rag/lessons/01-intro.md`, text search with `01-agentic-rag/lessons/14-agentic-loop.md`, and RRF with **01-agentic-rag/lessons/13-function-calling.md**.


In [14]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

print("RRF function ready")

RRF function ready


In [15]:
query = "How do I give the model access to tools?"
query_vector = embedder.encode(query)

vector_results = vindex.search(query_vector, num_results=5)
text_results = index.search(query, num_results=5)

results = rrf([vector_results, text_results])

print("Vector first:", vector_results[0]["filename"])
print("Text first:", text_results[0]["filename"])
print("RRF first:", results[0]["filename"])

Vector first: 01-agentic-rag/lessons/01-intro.md
Text first: 01-agentic-rag/lessons/14-agentic-loop.md
RRF first: 01-agentic-rag/lessons/13-function-calling.md


## Recorded answers and evidence boundary

| Question | Recorded answer | Saved observation |
| --- | --- | --- |
| Q1 | **-0.02** | First coordinate -0.020582046681707114 |
| Q2 | **0.37** | Cosine similarity 0.3610702427672444 |
| Q3 | **02-vector-search/lessons/07-sqlitesearch-vector.md** | Chunk 94; similarity 0.6489018052329629 |
| Q4 | **04-evaluation/lessons/05-search-metrics.md** | First vector result |
| Q5 | **02-vector-search/lessons/08-pgvector.md** | Present in vector top five, absent from text top five |
| Q6 | **01-agentic-rag/lessons/13-function-calling.md** | First result after RRF |

These are learner-run observations for one pinned dataset, model and dependency environment. They do not establish general search quality, production readiness or a submitted/graded course result. The Q2/Q3 score difference is specific to the page and chunk compared.

Code and outputs are retained from the learning session, including the equivalent `@` and `.dot()` calculations. Subsequent learning follows the lessons' sequence, variable names and notation, explaining necessary homework adaptations.
